# Week 2 Lab: KL by Hand, then an Autoencoder
**Deep Generative Models · GDS0058-1 · Week 2 (9/9)**

> **First: File ▸ Save a copy in Drive.** / 먼저 **파일 ▸ Drive에 사본 저장**을 누르세요. 그러지 않으면 작업 내용이 사라집니다.

**Today's flow (60 min)**
1. Walkthrough (15 min): the instructor runs sections 0 to 2 and points at the lines that match the slides.
2. Fill in the two `TODO` cells (15 min). Stuck? Expand the ▶ Solution cell below it and run that instead. That is a normal path, not a failure.
3. Solution review (10 min).
4. Experiment (15 min): change the latent size and see what changes.
5. Share (5 min): one result per person, one sentence.

한글 요약은 각 절 끝의 **"What to notice"** 아래에 한 줄씩 있습니다.

Everything runs on CPU. `numpy` only for the model; `torchvision` is used just to download MNIST (PyTorch models start in Week 4).


## 0. Setup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng = np.random.default_rng(SEED)


def load_mnist(n_train=10_000, n_test=2_000):
    # 1) torchvision (preinstalled on Colab)  2) fallback: keras copy of the same file
    try:
        from torchvision import datasets
        tr = datasets.MNIST("./data", train=True, download=True)
        te = datasets.MNIST("./data", train=False, download=True)
        Xtr, ytr = tr.data.numpy(), tr.targets.numpy()
        Xte, yte = te.data.numpy(), te.targets.numpy()
    except Exception as e:
        print("torchvision download failed, using keras copy:", e)
        from tensorflow.keras.datasets import mnist
        (Xtr, ytr), (Xte, yte) = mnist.load_data()
    Xtr = Xtr[:n_train].reshape(n_train, -1).astype(np.float32) / 255.0
    Xte = Xte[:n_test].reshape(n_test, -1).astype(np.float32) / 255.0
    return Xtr, ytr[:n_train], Xte, yte[:n_test]


Xtr, ytr, Xte, yte = load_mnist()
print("train", Xtr.shape, "test", Xte.shape, "| pixel range", Xtr.min(), Xtr.max())

fig, axes = plt.subplots(1, 10, figsize=(10, 1.3))
for k, ax in enumerate(axes):
    ax.imshow(Xtr[k].reshape(28, 28), cmap="gray"); ax.set_title(int(ytr[k])); ax.axis("off")
plt.show()


### Optional: Korean labels in plots / 그림에 한글 쓰기

Figures here are labelled in English, so you can skip this. But Colab ships **no Korean font**: the moment you write a Korean title yourself it comes out as boxes. The cell below fixes it for the session, **no runtime restart needed**.

이 노트북 그림은 영문이라 건너뛰어도 됩니다. 다만 Colab에는 한글 폰트가 없어서 여러분이 한글 제목을 쓰는 순간 네모로 깨집니다. 아래 셀이 **재시작 없이** 해결합니다.


In [ ]:
#@title ▶ 한글 폰트 설치 / Install Korean font (optional) { display-mode: "form" }
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

_p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
try:
    fm.fontManager.addfont(_p)                  # 캐시 재생성 없이 즉시 등록 -> 재시작 불필요
    plt.rc("font", family=fm.FontProperties(fname=_p).get_name())
    plt.rc("axes", unicode_minus=False)         # 음수 눈금 깨짐 방지
    print("Korean font ready:", plt.rcParams["font.family"][0])
except Exception as _e:
    # apt 가 막혀도 노트북 전체가 멈추지 않도록 한다. 그림 라벨만 영문/네모로 나온다.
    plt.rc("axes", unicode_minus=False)
    print("한글 폰트를 건너뜁니다 (그림 라벨이 깨질 수 있습니다):", _e)

# 함정 1. 라벨에 유니코드 마이너스(U+2212)나 공집합(U+2205)을 직접 타이핑하면
#         이 폰트에 글리프가 없어 네모로 뜹니다. ASCII 하이픈(-)을 쓰세요.
# 함정 2. 로그 축 눈금(10^-3 등)은 mathtext 로 그려지므로 unicode_minus=False 로도 막히지 않습니다.
#         로그 축을 쓸 때는 FuncFormatter 로 눈금 문자열을 직접 만드세요:
#           from matplotlib.ticker import FuncFormatter
#           ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"1e{int(round(np.log10(y)))}"))


---
## 1. KL divergence by hand
*Deck: “KL Divergence: Definition and Properties” → “Worked Example ③: KL Between Two Normals”*

For two 1-D Gaussians the KL divergence has a closed form:

$$\mathrm{KL}\left( N(\mu_1,\sigma_1^2) \parallel N(\mu_2,\sigma_2^2) \right) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1-\mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

In plain text, in case the formula does not render:

```
KL = log(s2 / s1) + (s1^2 + (m1 - m2)^2) / (2 * s2^2) - 1/2
```

This exact expression is the **KL term of the VAE loss in Weeks 3 to 4** (with $\mu_2 = 0, \sigma_2 = 1$). Write it once now and you will recognise it later.

Two checks follow: the slide numbers (0.125 / 0.5 / 2), and a numerical integral.


In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# Implement the closed form above. One line of arithmetic. Units: nats.
def kl_gauss(mu1, s1, mu2, s2):
    # KL( N(mu1, s1^2) || N(mu2, s2^2) )
    raise NotImplementedError(
        "\n\n"
        "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
        "     위 수식을 한 줄로 옮겨 적고 이 셀을 다시 실행하세요.\n"
        "     막히면 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
        "  This is expected, not a broken notebook.\n"
        "     Write the closed form above as one line and re-run, or run the Solution cell just below.\n"
    )


In [ ]:
#@title ▶ Solution / 정답 (run only if you are stuck; this overwrites your version) { display-mode: "form" }
def kl_gauss(mu1, s1, mu2, s2):
    return np.log(s2 / s1) + (s1**2 + (mu1 - mu2)**2) / (2 * s2**2) - 0.5


In [ ]:
# Check 0: the three numbers from the “Worked Example ③: KL Between Two Normals” slide
#          (equal variances, mu = 0.5, 1, 2  ->  0.125, 0.5, 2)
for mu in (0.5, 1.0, 2.0):
    print(f"KL( N({mu},1) || N(0,1) ) = {kl_gauss(mu, 1.0, 0.0, 1.0):.3f}")

# Check 1: closed form vs numerical integration of  p(x) log(p(x)/q(x))
x = np.linspace(-15, 15, 200_001)
def npdf(x, mu, s): return np.exp(-0.5 * ((x - mu) / s)**2) / (s * np.sqrt(2 * np.pi))
p, q = npdf(x, 0, 1), npdf(x, 1, 2)
numeric = (p * np.log(p / q)).sum() * (x[1] - x[0])
print(f"closed form {kl_gauss(0, 1, 1, 2):.6f}   numeric {numeric:.6f}")

# Check 2: KL is not symmetric
print(f"KL(p||q) = {kl_gauss(0, 1, 1, 2):.4f}   KL(q||p) = {kl_gauss(1, 2, 0, 1):.4f}")

# Picture: how KL to the standard normal grows as the mean drifts (this is what the VAE will penalise)
mus = np.linspace(-3, 3, 121)
plt.figure(figsize=(5, 3))
plt.plot(mus, kl_gauss(mus, 1.0, 0.0, 1.0), label="σ = 1")
plt.plot(mus, kl_gauss(mus, 0.5, 0.0, 1.0), label="σ = 0.5")
plt.xlabel("μ of the first Gaussian"); plt.ylabel("KL to N(0, 1)  [nats]"); plt.legend(); plt.tight_layout(); plt.show()


**What to notice.** The slide numbers come out, and the closed form agrees with the integral to 5 or 6 decimals, so the formula is right. The asymmetry is real, and it matters from Week 3 on (forward vs reverse KL). The curve is the "pull toward the prior" that a VAE feels. An autoencoder feels nothing of the kind, which is the point of section 3.

> 요약: 수식이 맞는지 슬라이드 숫자와 수치적분으로 확인했다. KL은 방향에 따라 값이 다르고, 평균이 0에서 멀어질수록 KL이 커진다. 이 "끌어당김"이 VAE에는 있고 오토인코더에는 없다.


---
## 2. An autoencoder in numpy
*Deck: “The Autoencoder”, “What an AE Does Well and What It Cannot Do”*

Architecture: `784 → 128 → d_z → 128 → 784`, ReLU inside, sigmoid at the output so pixels land in [0, 1].
The backward pass is written by hand below so you can see that there is nothing hidden. **You do not need to read `backward`**; from Week 4 PyTorch does this for us.

The only line that is *the model's idea* is the loss: reconstruction error $\lVert x-\hat{x} \rVert^2$ (squared distance between input and output). That is your TODO.


In [ ]:
def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-np.clip(a, -30, 30)))


class Autoencoder:
    def __init__(self, d_in=784, h=128, d_z=2, seed=SEED):
        r = np.random.default_rng(seed)
        he = lambda a, b: r.normal(0, np.sqrt(2 / a), (a, b)).astype(np.float32)
        self.p = {"W1": he(d_in, h), "b1": np.zeros(h, np.float32),      # encoder
                  "W2": he(h, d_z), "b2": np.zeros(d_z, np.float32),     # bottleneck
                  "W3": he(d_z, h), "b3": np.zeros(h, np.float32),       # decoder
                  "W4": he(h, d_in), "b4": np.zeros(d_in, np.float32)}
        # Adam state
        self.m = {k: np.zeros_like(v) for k, v in self.p.items()}
        self.v = {k: np.zeros_like(v) for k, v in self.p.items()}
        self.t = 0

    def encode(self, x):
        p = self.p
        return np.maximum(0, x @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]

    def decode(self, z):
        p = self.p
        return sigmoid(np.maximum(0, z @ p["W3"] + p["b3"]) @ p["W4"] + p["b4"])

    def forward(self, x):                       # same as encode→decode, but keeps intermediates
        p = self.p
        self.x = x
        self.a1 = x @ p["W1"] + p["b1"];  self.h1 = np.maximum(0, self.a1)
        self.z = self.h1 @ p["W2"] + p["b2"]
        self.a2 = self.z @ p["W3"] + p["b3"]; self.h2 = np.maximum(0, self.a2)
        self.xhat = sigmoid(self.h2 @ p["W4"] + p["b4"])
        return self.xhat

    def backward(self, dxhat):                  # manual backprop; returns gradients
        p, g = self.p, {}
        da4 = dxhat * self.xhat * (1 - self.xhat);       g["W4"] = self.h2.T @ da4; g["b4"] = da4.sum(0)
        da2 = (da4 @ p["W4"].T) * (self.a2 > 0);         g["W3"] = self.z.T @ da2;  g["b3"] = da2.sum(0)
        dz = da2 @ p["W3"].T;                            g["W2"] = self.h1.T @ dz;  g["b2"] = dz.sum(0)
        da1 = (dz @ p["W2"].T) * (self.a1 > 0);          g["W1"] = self.x.T @ da1;  g["b1"] = da1.sum(0)
        return g

    def adam_step(self, g, lr=3e-3, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for k in self.p:
            self.m[k] = b1 * self.m[k] + (1 - b1) * g[k]
            self.v[k] = b2 * self.v[k] + (1 - b2) * g[k] ** 2
            self.p[k] -= lr * (self.m[k] / (1 - b1 ** self.t)) / (np.sqrt(self.v[k] / (1 - b2 ** self.t)) + eps)


In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# Reconstruction loss and its gradient with respect to xhat.
#   loss  = mean over the batch of  sum over pixels of (xhat - x)^2      (one number)
#   dxhat = d loss / d xhat                                              (same shape as xhat)
#
#   주의: loss 가 배치에 대한 '평균'이므로 dxhat 에도 1/B 가 들어갑니다 (B = x.shape[0]).
#     이걸 빠뜨려도 Adam 이 기울기 크기에 거의 불변이라 학습이 정상으로 보입니다 - 조용히 틀립니다.
#     아래 자가검증 셀이 그것을 잡아 줍니다.
def recon_loss(x, xhat):
    raise NotImplementedError(
        "\n\n"
        "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
        "     loss와 dxhat 두 줄을 채우고 이 셀을 다시 실행하세요.\n"
        "     막히면 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
        "  This is expected, not a broken notebook.\n"
        "     Fill in loss and dxhat (two lines) and re-run, or run the Solution cell just below.\n"
    )
    return loss, dxhat


In [ ]:
#@title ▶ Solution / 정답 (run only if you are stuck) { display-mode: "form" }
def recon_loss(x, xhat):
    loss = ((xhat - x) ** 2).sum(axis=1).mean()
    dxhat = 2 * (xhat - x) / x.shape[0]
    return loss, dxhat


In [ ]:
# 자가검증 / Check your work — recon_loss 의 값과 기울기를 수치미분과 대조
_r = np.random.default_rng(0)
_x  = _r.normal(0, 1, (5, 4))
_xh = _r.normal(0, 1, (5, 4))
_l, _g = recon_loss(_x, _xh)
_expect = ((_xh - _x) ** 2).sum(axis=1).mean()
assert abs(_l - _expect) < 1e-10, f"loss 가 {_l}, 기대값 {_expect}"
_eps = 1e-6; _num = np.zeros_like(_xh)
for _i in range(_xh.shape[0]):
    for _j in range(_xh.shape[1]):
        _p = _xh.copy(); _p[_i, _j] += _eps
        _m = _xh.copy(); _m[_i, _j] -= _eps
        _num[_i, _j] = (recon_loss(_x, _p)[0] - recon_loss(_x, _m)[0]) / (2 * _eps)
assert np.allclose(_g, _num, atol=1e-6), (
    "dxhat 이 수치미분과 다릅니다. loss 가 배치 평균이므로 1/B 를 곱해야 합니다 "
    f"(최대 오차 {np.abs(_g - _num).max():.3e})"
)
print("OK  loss =", round(float(_l), 6), " | dxhat 최대오차 =", f"{np.abs(_g - _num).max():.2e}")


In [ ]:
def train_ae(d_z, epochs=15, lr=3e-3, batch=128, seed=SEED, verbose=True):
    ae = Autoencoder(d_z=d_z, seed=seed)
    r = np.random.default_rng(seed)
    history = []
    t0 = time.time()
    for ep in range(epochs):
        idx = r.permutation(len(Xtr)); total = 0.0
        for i in range(0, len(Xtr), batch):
            xb = Xtr[idx[i:i + batch]]
            xhat = ae.forward(xb)
            loss, dxhat = recon_loss(xb, xhat)
            ae.adam_step(ae.backward(dxhat), lr=lr)
            total += loss * len(xb)
        history.append(total / len(Xtr))
        if verbose and (ep % 5 == 4 or ep == 0):
            print(f"epoch {ep+1:2d}  train loss {history[-1]:6.1f}")
    test_loss = ((ae.decode(ae.encode(Xte)) - Xte) ** 2).sum(1).mean()
    print(f"d_z={d_z}: test reconstruction error {test_loss:.1f} per image  ({time.time()-t0:.0f}s)")
    return ae, history, test_loss


ae2, hist2, test2 = train_ae(d_z=2)

plt.figure(figsize=(5, 3)); plt.plot(range(1, len(hist2) + 1), hist2, marker="o")
plt.xlabel("epoch"); plt.ylabel("train reconstruction error"); plt.title("d_z = 2"); plt.tight_layout(); plt.show()


In [ ]:
def show_pairs(ae, X, n=10, title=""):
    xhat = ae.decode(ae.encode(X[:n]))
    fig, axes = plt.subplots(2, n, figsize=(n, 2.3))
    for k in range(n):
        axes[0, k].imshow(X[k].reshape(28, 28), cmap="gray");    axes[0, k].axis("off")
        axes[1, k].imshow(xhat[k].reshape(28, 28), cmap="gray"); axes[1, k].axis("off")
    axes[0, 0].set_title("input", fontsize=9, loc="left"); axes[1, 0].set_title("reconstruction", fontsize=9, loc="left")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_pairs(ae2, Xte, title="d_z = 2  (test images)")


**What to notice.** The loss drops fast and then flattens. Two numbers cannot hold a 784-pixel image, so the decoder returns the "average digit of that region": blurry but recognisable. The bottleneck is doing exactly what the slide said, keeping only what matters.

> 요약: 784픽셀을 숫자 2개로 줄였다가 복원하면 흐릿하지만 알아볼 수는 있다. 병목이 "중요한 것만 남기는" 압축을 강제한다.


---
## 3. The latent scatter
*Deck: “Board Example: the Latent Scatter”*


In [ ]:
Z = ae2.encode(Xte)
plt.figure(figsize=(6, 5.5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=yte, cmap="tab10", s=6)
plt.colorbar(sc, ticks=range(10), label="digit")
plt.xlabel("z[0]"); plt.ylabel("z[1]"); plt.title("AE latent space (d_z = 2, test set)"); plt.tight_layout(); plt.show()
print("latent range  z[0]: %.1f .. %.1f   z[1]: %.1f .. %.1f" % (Z[:, 0].min(), Z[:, 0].max(), Z[:, 1].min(), Z[:, 1].max()))


**What to notice.** Digits cluster without ever seeing a label: this is *representation learning*. Now look at the empty regions and at the scale of the axes. Nothing told the encoder where to put things or how far to spread them. Section 4 shows what that costs.

> 요약: 라벨 없이도 숫자별로 뭉친다(표현학습). 하지만 군집 사이 빈 공간과 축의 범위는 아무도 정해 주지 않았다. 이것이 다음 절에서 문제가 된다.


---
## 4. Decoding a random z
*Deck: “What an AE Does Well and What It Cannot Do” → “Common Pitfalls”*

If the latent space were a proper generative latent, any $z$ we draw should decode to a plausible digit. Try two ways of drawing $z$.


In [ ]:
def show_grid(imgs, title):
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=(n, 1.3))
    for k, ax in enumerate(axes):
        ax.imshow(imgs[k].reshape(28, 28), cmap="gray"); ax.axis("off")
    fig.suptitle(title, fontsize=10); plt.tight_layout(); plt.show()

# 이 셀만 다시 돌려도 같은 그림이 나오도록 셀 안에서 난수를 초기화한다.
# (모듈 최상단의 rng 를 쓰면 재실행할 때마다 z 가 달라진다.)
rng = np.random.default_rng(SEED)

# (a) z ~ N(0, I): the distribution a VAE will *force* its latent to follow (Week 4)
z_gauss = rng.normal(size=(12, 2)).astype(np.float32)
show_grid(ae2.decode(z_gauss), "(a) z ~ N(0, I)")

# (b) z uniform over the box the encoder actually used
lo, hi = Z.min(0), Z.max(0)
z_box = rng.uniform(lo, hi, size=(12, 2)).astype(np.float32)
show_grid(ae2.decode(z_box), "(b) z uniform inside the encoder's own bounding box")

# (c) z on a straight line between two real test images (interpolation)
za, zb = ae2.encode(Xte[:1]), ae2.encode(Xte[1:2])
z_line = (za + (zb - za) * np.linspace(0, 1, 12)[:, None]).astype(np.float32)
show_grid(ae2.decode(z_line), f"(c) interpolation  {int(yte[0])} → {int(yte[1])}")


**What to notice.** (a) is often nonsense: N(0, I) may not even overlap the region the encoder used. (b) is better but hits the holes: some samples are blends of two digits, some are smears. (c) works best because it stays on a path between real points. **An autoencoder compresses; it does not generate.** Weeks 3 to 4 fix this by putting a prior on $z$, and the price of that prior is the KL term you wrote in section 1.

> 요약: 임의의 z를 넣으면 (a) N(0,I)는 인코더가 쓰지 않은 영역이라 엉망, (b) 인코더 범위 안에서 뽑아도 구멍에 걸리면 뭉개짐, (c) 실제 두 점 사이 보간만 그럭저럭. 오토인코더는 압축은 하지만 생성은 못 한다. VAE는 z에 사전분포를 씌워 이를 고치고, 그 대가가 1절의 KL 항이다.


---
## 5. Experiment: how big should the bottleneck be? (15 min)

Train `d_z = 8` and `d_z = 32` and compare with `d_z = 2`. Then answer the three questions below in the last cell. Pick one of the questions to think about while it trains.


In [ ]:
results = {2: test2}
models = {2: ae2}
for d_z in (8, 32):
    ae_d, hist_d, test_d = train_ae(d_z=d_z, verbose=False)
    models[d_z] = ae_d; results[d_z] = test_d

print("\nd_z   test reconstruction error per image")
for d_z, err in sorted(results.items()):
    print(f"{d_z:3d}   {err:6.1f}")

for d_z in (8, 32):
    show_pairs(models[d_z], Xte, title=f"d_z = {d_z}  (test images)")


**Questions for the last cell**
1. Reconstruction error vs `d_z`: is the gain from 2 → 8 larger or smaller than from 8 → 32? Why does it flatten?
2. `d_z = 32` reconstructs almost perfectly. Can you still *plot* its latent space? What did you lose?
3. If you had to pick one `d_z` for *your* data (images, tables, signals), what would you trade for what?


---
## 6. 더 해보기 / Going further (optional, not graded)

- **Denoising.** Add noise to the *input* only (`xb + 0.3 * rng.normal(size=xb.shape)`), keep the clean image as the target. Does the latent scatter get tighter?
- **FashionMNIST.** Replace `datasets.MNIST` with `datasets.FashionMNIST`. Which classes collapse together in 2-D?
- **Bigger hidden layer.** `h=256`. Does it help `d_z = 2` at all?


In [ ]:
# Scratch space for section 6.


---
## 7. Last cell: three lines + environment (the required format for assignments; practise it now)

> 과제 제출 형식 연습: 바꾼 것, 핵심 수치, 한 줄 해석. 실행 환경과 시드도 함께 남긴다.


In [ ]:
import sys, platform, datetime
print("Runtime :", platform.platform(), "| Python", sys.version.split()[0])
print("numpy   :", np.__version__)
print("Run at  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Seed    :", SEED)

summary = {
    "What I changed": "...",           # e.g. d_z 2 → 8 → 32
    "Main number":    "...",           # e.g. test error 36.6 / 19.5 / 9.2
    "One-line take":  "...",           # e.g. compression buys plottability, costs fidelity
}
for k, v in summary.items():
    print(f"{k:15s}: {v}")
